In [0]:
%sql
CREATE CATALOG IF NOT EXISTS telecom_catalog_assign;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;

In [0]:
%sql
SHOW CATALOGS;
SHOW SCHEMAS IN telecom_catalog_assign;
SHOW VOLUMES IN telecom_catalog_assign.landing_zone;

In [0]:
dbutils.fs.mkdirs("/Workspace/Users/nithish.t1712@gmail.com/DWH-REPO/Databricks/TELECOM_PROJ/source_files/")

In [0]:
base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

for folder in ["customer", "usage", "tower"]:
    dbutils.fs.mkdirs(f"{base}/{folder}/")

display(dbutils.fs.ls(base))

In [0]:
base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

dbutils.fs.mkdirs(f"{base}/tower/region1/")
dbutils.fs.mkdirs(f"{base}/tower/region2/")

display(dbutils.fs.ls(f"{base}/tower/"))

In [0]:
import os

catalog, schema, volume = "telecom_catalog_assign", "landing_zone", "landing_vol"
base = f"/Volumes/{catalog}/{schema}/{volume}"
folders = ["customer", "usage", "tower"]
tower_regions = ["region1", "region2"]

src_file = "/Workspace/Users/nithish.t1712@gmail.com/DWH-REPO/Databricks/TELECOM_PROJ/source_files/"

def check(label, ok):
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
    return ok

def list_names(path):
    """Return the folder and file names inside a path, or [] if it doesn't exist."""
    try:
        return [f.name.rstrip("/") for f in dbutils.fs.ls(path)]
    except Exception:
        return []
    
results = []

# 1. catalog, schema, volume
results.append(check(f"Catalog {catalog}",
    spark.sql(f"SHOW CATALOGS LIKE '{catalog}'").count() == 1))
results.append(check(f"Schema {catalog}.{schema}",
    spark.sql(f"SHOW SCHEMAS IN {catalog} LIKE '{schema}'").count() == 1))
results.append(check(f"Volume {catalog}.{schema}.{volume}",
    spark.sql(f"SHOW VOLUMES IN {catalog}.{schema} LIKE '{volume}'").count() == 1))

# 2. folders inside the volume
try:
    existing = [f.name.rstrip("/") for f in dbutils.fs.ls(base)]
except Exception:
    existing = []

for folder in folders:
    results.append(check(f"Volume folder {base}/{folder}/", folder in existing))

# 3. region folders under tower
tower_existing = list_names(f"{base}/tower")
for region in tower_regions:
    results.append(check(f"Region folder {base}/tower/{region}/", region in tower_existing))


# 4. source file folder in the workspace
results.append(check(f"Source folder {src_file}", os.path.isdir(src_file)))

if os.path.isdir(src_file):
    print("\nFiles in the source folder:")
    for name in sorted(os.listdir(src_file)):
        size = os.path.getsize(os.path.join(src_file, name))
        print(f"  {name}  ({size} bytes)")

print("\nAll checks passed" if all(results) else "\nSome checks failed")

In [0]:
src_file = "/Workspace/Users/nithish.t1712@gmail.com/DWH-REPO/Databricks/TELECOM_PROJ/source_files/"
base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

# source file name -> destination folder in the volume
files_to_copy = {
    "customer.csv":           f"{base}/customer/",
    "usage.tsv":              f"{base}/usage/",
    "tower_logs_region1.txt": f"{base}/tower/region1/",
    "tower_logs_region2.txt": f"{base}/tower/region2/",
}

for file_name, dest_folder in files_to_copy.items():
    # read the source file as text
    with open(f"{src_file}{file_name}", "r", encoding="utf-8") as f:
        content = f.read()

    # write it into the volume folder
    dbutils.fs.put(f"{dest_folder}{file_name}", content, overwrite=True)
    print(f"Copied {file_name} -> {dest_folder}")

for dest_folder in files_to_copy.values():
    display(dbutils.fs.ls(dest_folder))


In [0]:
base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

customer = spark.read.csv(f"{base}/customer/customer.csv",
    schema="customer_id INT, name STRING, age INT, city STRING, plan_type STRING")
customer.show(2)

usage = (spark.read.options(header=True, sep="\t")
    .schema("customer_id INT, voice_mins INT, data_mb INT, sms_count INT")
    .csv(f"{base}/usage/usage.tsv"))

tower_logs = (spark.read.options(header=True, sep="|")
    .schema("event_id STRING, customer_id INT, tower_id STRING, signal_strength INT, timestamp TIMESTAMP")
    .csv(f"{base}/tower/tower_logs_region1.txt"))

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"
out = "/Volumes/telecom_catalog_assign/landing_zone/output/output"

cust_schema=StructType([
    StructField("CustId",IntegerType(),True),
    StructField("FullName",StringType(),False),
    StructField("Age",IntegerType(),False),
    StructField("City",StringType(),False),
    StructField("PlanType",StringType(),False)
])
cust_df=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",schema=cust_schema,header=False,sep=',',pathGlobFilter="*.csv")
print(cust_df.count())
cust_df.printSchema()

# Customer is the DataFrame you read earlier
cust_df.write.mode("overwrite").parquet(f"{out}/customer_parquet")
cust_df.write.mode("overwrite").json(f"{out}/customer_json")
cust_df.write.mode("overwrite").orc(f"{out}/customer_orc")
cust_df.write.mode("overwrite").option("header", True).csv(f"{out}/customer_csv")


usage_schem=StructType([
    StructField("CustId",IntegerType(),True),
    StructField("Usage_mins",IntegerType(),False),
    StructField("Data_MB",IntegerType(),False),
    StructField("SMS_Count",IntegerType(),False)
])

usage = (spark.read.options(header=True, sep="\t")
    .schema(usage_schem)
    .csv(f"{base}/usage/usage.tsv"))

print(usage.count())
usage.printSchema()

# Usage is the DataFrame you read earlier
usage.write.mode("overwrite").parquet(f"{out}/usage_parquet")
usage.write.mode("overwrite").json(f"{out}/usage_json")
usage.write.mode("overwrite").orc(f"{out}/usage_orc")
usage.write.mode("overwrite").option("header", True).csv(f"{out}/usage_csv")


tower_schema=StructType([
    StructField("EventId",StringType(),False),
    StructField("CustId",IntegerType(),True),
    StructField("TowerId",StringType(),False),
    StructField("Signal",IntegerType(),False),
    StructField("Timestamp",StringType(),False)
])
paths = [f"{base}/tower/region1/",
    f"{base}/tower/region2/",
]

tower_logs = (spark.read
    .options(header=True, sep="|", pathGlobFilter="*.txt", recursiveFileLookup=True)
    .schema(tower_schema)
    .csv(paths))

print(tower_logs.count())
tower_logs.printSchema()

# tower_logs is the DataFrame you read earlier
tower_logs.write.mode("overwrite").parquet(f"{out}/tower_parquet")
tower_logs.write.mode("overwrite").json(f"{out}/tower_json")
tower_logs.write.mode("overwrite").orc(f"{out}/tower_orc")
tower_logs.write.mode("overwrite").option("header", True).csv(f"{out}/tower_csv")

In [0]:
base = "/Volumes/telecom_catalog_assign/landing_zone/output"

# 1. Read the ORC data (written earlier) into a DataFrame
df_cust = spark.read.orc(f"{base}/output/customer_orc")
df_usage = spark.read.orc(f"{base}/output/usage_orc")
df_tower = spark.read.orc(f"{base}/output/tower_orc")

df_cust.printSchema()
#df_cust.show(5)
print(df_cust.count())


df_usage.printSchema()
#df_usage.show(5)
print(df_usage.count())

df_tower.printSchema()
#df_tower.show(5)
print(df_tower.count())

# 2. Write the DataFrame to DBFS in Parquet format
dbfs_path = "dbfs:/Volumes/telecom_catalog_assign/landing_zone/output_parquet"

df_cust.write.mode("overwrite").parquet(f"{dbfs_path}/customer_parquet")
df_usage.write.mode("overwrite").parquet(f"{dbfs_path}/usage_parquet")
df_tower.write.mode("overwrite").parquet(f"{dbfs_path}/tower_parquet")

# 3. Verify
display(dbutils.fs.ls(dbfs_path))
spark.read.parquet(f"{dbfs_path}/customer_parquet").show(5)
spark.read.parquet(f"{dbfs_path}/usage_parquet").show(5)
spark.read.parquet(f"{dbfs_path}/tower_parquet").show(5)